In [ ]:
import os
import json
import time
import random
from datetime import datetime
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import StructType, StructField, StringType, DoubleType

# Configuración de rutas absolutas locales dentro del contenedor
BRONZE_PATH_LOCAL = "/home/jovyan/work/lakehouse/bronze/"
BRONZE_PATH_SPARK = "file:///home/jovyan/work/lakehouse/bronze/"
SILVER_PATH_SPARK = "file:///home/jovyan/work/lakehouse/silver/telemetry_clean"
ALERT_LOG_PATH = "/home/jovyan/work/lakehouse/alertas.log"

# Asegurar que los directorios físicos existan en el contenedor
os.makedirs(BRONZE_PATH_LOCAL, exist_ok=True)

# ==========================================
# ETAPA 1: INGESTA (Simulación y escritura local)
# ==========================================
print("🚀 1. INGESTA: Generando ráfagas de datos IoT...")
SENSORES_CONFIG = {
    "SENSOR_01": {"temp_base": 42.0, "volt_base": 220.0, "vib_base": 0.05},
    "SENSOR_02": {"temp_base": 55.0, "volt_base": 380.0, "vib_base": 0.12}
}

# Generamos 3 lotes rápidos para tener datos reales que procesar
for ciclo in range(3):
    lote_datos = []
    for _ in range(200): # 200 lecturas por lote
        sensor_id = random.choice(list(SENSORES_CONFIG.keys()))
        config = SENSORES_CONFIG[sensor_id]
        prob = random.random()
        ts = datetime.now().isoformat()
        
        if prob < 0.04: # Inyección de Nulos (Ruido matemático para que la IA aprenda)
            lote_datos.append({"timestamp": ts, "sensor_id": sensor_id, "temperatura": None, "voltaje": config["volt_base"], "vibracion": None, "estado_alerta": "SIGNAL_LOSS"})
        else: # Datos Normales
            lote_datos.append({"timestamp": ts, "sensor_id": sensor_id, "temperatura": round(config["temp_base"] + random.uniform(-1, 1), 2), "voltaje": round(config["volt_base"] + random.uniform(-2, 2), 2), "vibracion": round(config["vib_base"], 4), "estado_alerta": "OK"})
            if prob > 0.95: # Forzar Duplicados alternos
                lote_datos.append(lote_datos[-1])

    with open(os.path.join(BRONZE_PATH_LOCAL, f"batch_{ciclo}.json"), 'w') as f:
        json.dump(lote_datos, f)

print(f"📁 Archivos creados en Bronze: {os.listdir(BRONZE_PATH_LOCAL)}")

# ==========================================
# ETAPA 2: CARGA (Inicializar Spark y leer JSONs)
# ==========================================
print("\n🤖 2. CARGA: Inicializando sesión de Spark...")
spark = SparkSession.builder \
    .appName("IoT-Lakehouse-Silver") \
    .config("spark.jars.packages", "io.delta:delta-spark_2.12:3.2.0") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .getOrCreate()

schema = StructType([
    StructField("timestamp", StringType(), True),
    StructField("sensor_id", StringType(), True),
    StructField("temperatura", DoubleType(), True),
    StructField("voltaje", DoubleType(), True),
    StructField("vibracion", DoubleType(), True),
    StructField("estado_alerta", StringType(), True)
])

# Spark lee la carpeta de forma directa (Capa BRONZE real)
df_raw = spark.read.schema(schema).json(BRONZE_PATH_SPARK)
total_bronze = df_raw.count()
print(f"📥 Registros crudos cargados en el DataFrame: {total_bronze}")

# ==========================================
# ETAPA 3: METRICAS DATAOPS - CÁLCULO DEL KPI: DER
# ==========================================
print("\n📈 METRICAS: Calculando KPI Tasa de Error de Datos (DER)...")
# Filtramos registros corruptos usando los nombres de columna correctos del esquema
df_corruptos = df_raw.filter("vibracion IS NULL OR temperatura IS NULL")
registros_corruptos = df_corruptos.count()

der = (registros_corruptos / total_bronze) * 100 if total_bronze > 0 else 0
print(f"   -> KPI - Tasa de Error de Datos (DER): {der:.2f}%")

# Alerta persistente si el error supera el umbral del 1.5% regulatorio
if der > 1.5:
    ts_actual = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    with open(ALERT_LOG_PATH, "a") as f:
        f.write(f"[{ts_actual}] ALERTA CRÍTICA: DER de {der:.2f}% supera el umbral de 1.5%\n")
    print("   ⚠️ Alerta crítica registrada en el archivo log.")

# ==========================================
# ETAPA 4: TRANSFORMACIÓN (Limpieza profunda)
# ==========================================
print("\n🔄 4. TRANSFORMACIÓN: Removiendo duplicados e imputando nulos...")
df_deduplicated = df_raw.dropDuplicates(["timestamp", "sensor_id"])

# Ventana para promediar métricas por cada sensor individual (Imputación inteligente)
window_sensor = Window.partitionBy("sensor_id")

df_transformed = df_deduplicated \
    .withColumn("temperatura_limpia", F.coalesce(F.col("temperatura"), F.avg("temperatura").over(window_sensor))) \
    .withColumn("vibracion_limpia", F.coalesce(F.col("vibracion"), F.avg("vibracion").over(window_sensor))) \
    .drop("temperatura", "vibracion")

# ==========================================
# ETAPA 5: METRICAS DATAOPS - CÁLCULO DEL KPI: DCR
# ==========================================
print("\n📈 METRICAS: Calculando KPI Tasa de Completitud del Esquema (DCR)...")
# total_silver debe calcularse ANTES de usar df_transformed.count()
total_silver = df_transformed.count()

# DCR mide el porcentaje de datos completos tras la imputación inteligente (coalesce)
dcr = (df_transformed.filter("temperatura_limpia IS NOT NULL AND vibracion_limpia IS NOT NULL").count() / total_silver) * 100

print(f"   -> KPI - Tasa de Completitud (DCR): {dcr:.2f}%")

# ==========================================
# ETAPA 6: VALIDACIÓN Y ESCRITURA FINAL (Capa SILVER Delta)
# ==========================================
print("\n✅ 6. VALIDACIÓN: Tipado estricto y volcado a Delta Lake...")
df_validated = df_transformed \
    .withColumn("timestamp", F.to_timestamp("timestamp")) \
    .withColumn("voltaje", F.when(F.col("voltaje") < 0, 0.0).otherwise(F.col("voltaje")))

# Escritura analítica atómica en Silver (Mantenemos overwrite o append según el diseño original)
df_validated.write.format("delta").mode("overwrite").save(SILVER_PATH_SPARK)
print(f"💾 ¡Pipeline completado! Datos validados y resguardados en Silver: {SILVER_PATH_SPARK}")

🚀 1. INGESTA: Generando ráfagas de datos IoT...
📁 Archivos creados en Bronze: ['batch_0.json', 'batch_1.json', 'batch_2.json']

🤖 2. CARGA: Inicializando sesión de Spark...
📥 Registros crudos cargados en el DataFrame: 625

🔄 3. TRANSFORMACIÓN: Removiendo duplicados e imputando nulos...

✅ 4. VALIDACIÓN: Tipado estricto y volcado a Delta Lake...
💾 ¡Pipeline completado! Datos validados y resguardados en Silver: file:///home/jovyan/work/lakehouse/silver/telemetry_clean
